In [24]:
import pandas as pd
import numpy as np
train = pd.read_csv("/content/train_v9rqX0R.csv")
test = pd.read_csv("/content/test_AbJTz2l.csv")

In [25]:
train['source'] = 'train'
test['source'] = 'test'

test['Item_Outlet_Sales'] = 0  # placeholder

data = pd.concat([train, test], ignore_index=True)
data.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,source
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380,train
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228,train
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700,train
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800,train
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052,train


In [26]:
data.isnull().sum()

,0
Item_Identifier,0
Item_Weight,2439
Item_Fat_Content,0
Item_Visibility,0
Item_Type,0
Item_MRP,0
Outlet_Identifier,0
Outlet_Establishment_Year,0
Outlet_Size,4016
Outlet_Location_Type,0


In [27]:
data['Item_Fat_Content'] = data['Item_Fat_Content'].replace({
    'LF': 'Low Fat',
    'low fat': 'Low Fat',
    'reg': 'Regular'
})

In [28]:
item_weight_avg = data.groupby('Item_Identifier')['Item_Weight'].mean()

data['Item_Weight'] = data['Item_Weight'].fillna(
    data['Item_Identifier'].map(item_weight_avg)
)

In [29]:
data['Outlet_Size'] = data['Outlet_Size'].fillna('Medium')


In [30]:
visibility_avg = data['Item_Visibility'].mean()
data['Item_Visibility'] = data['Item_Visibility'].fillna(visibility_avg)
visibility_avg

np.float64(0.06595278007399324)

In [31]:
visibility_avg = data.groupby('Item_Identifier')['Item_Visibility'].mean()

data['Item_Visibility'] = data.apply(
    lambda x: visibility_avg[x['Item_Identifier']]
    if x['Item_Visibility'] == 0 else x['Item_Visibility'],
    axis=1
)
visibility_avg.mean()

np.float64(0.06568950698153324)

In [32]:
data['Item_Type_Combined'] = data['Item_Identifier'].apply(lambda x: x[:2])

data['Item_Type_Combined'] = data['Item_Type_Combined'].map({
    'FD': 'Food',
    'NC': 'Non-Consumable',
    'DR': 'Drinks'
})
data.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,source,Item_Type_Combined
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380,train,Food
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228,train,Drinks
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700,train,Food
3,FDX07,19.20,Regular,0.017834,Fruits and Vegetables,182.0950,OUT010,1998,Medium,Tier 3,Grocery Store,732.3800,train,Food
4,NCD19,8.93,Low Fat,0.009780,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052,train,Non-Consumable


In [33]:
data['Outlet_Years'] = 2013 - data['Outlet_Establishment_Year']

In [34]:
data.loc[data['Item_Type_Combined'] == 'Non-Consumable', 'Item_Fat_Content'] = 'Non-Edible'

In [35]:
data.drop(['Item_Type', 'Outlet_Establishment_Year'], axis=1, inplace=True)

In [36]:
data = pd.get_dummies(data, columns=[
    'Item_Fat_Content',
    'Outlet_Location_Type',
    'Outlet_Size',
    'Outlet_Type',
    'Item_Type_Combined'
])

In [37]:
data.head()

,Item_Identifier,Item_Weight,Item_Visibility,Item_MRP,Outlet_Identifier,Item_Outlet_Sales,source,Outlet_Years,Item_Fat_Content_Low Fat,Item_Fat_Content_Non-Edible,...,Outlet_Size_High,Outlet_Size_Medium,Outlet_Size_Small,Outlet_Type_Grocery Store,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3,Item_Type_Combined_Drinks,Item_Type_Combined_Food,Item_Type_Combined_Non-Consumable
0,FDA15,9.30,0.016047,249.8092,OUT049,3735.1380,train,14,True,False,...,False,True,False,False,True,False,False,False,True,False
1,DRC01,5.92,0.019278,48.2692,OUT018,443.4228,train,4,False,False,...,False,True,False,False,False,True,False,True,False,False
2,FDN15,17.50,0.016760,141.6180,OUT049,2097.2700,train,14,True,False,...,False,True,False,False,True,False,False,False,True,False
3,FDX07,19.20,0.017834,182.0950,OUT010,732.3800,train,15,False,False,...,False,True,False,True,False,False,False,False,True,False
4,NCD19,8.93,0.009780,53.8614,OUT013,994.7052,train,26,False,True,...,True,False,False,False,True,False,False,False,False,True


In [38]:

data = data.replace({True: 1, False: 0})

/tmp/ipykernel_6230/3269834321.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.replace({True: 1, False: 0})


In [39]:
train_data = data[data['source'] == 'train']
test_data = data[data['source'] == 'test']
# Define X and y
X = train_data.drop('Item_Outlet_Sales', axis=1)
y = train_data['Item_Outlet_Sales']

# Train-test split (for validation)
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_val.shape)

(6818, 23) (1705, 23)


In [40]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

# Drop non-numeric columns 'Item_Identifier', 'Outlet_Identifier', and 'source'
X_train_numeric = X_train.drop(['Item_Identifier', 'Outlet_Identifier', 'source'], axis=1)
X_val_numeric = X_val.drop(['Item_Identifier', 'Outlet_Identifier', 'source'], axis=1)

rf.fit(X_train_numeric, y_train)

RandomForestRegressor(max_depth=10, min_samples_leaf=2, min_samples_split=5,
                      n_estimators=300, n_jobs=-1, random_state=42)

In [41]:
from sklearn.metrics import mean_squared_error
import numpy as np

y_pred = rf.predict(X_val_numeric)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print("RMSE:", rmse)

RMSE: 1034.7371360457735


In [42]:
rf.score(X_val_numeric, y_val)

0.6060733141020304

In [43]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=4,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

gbr.fit(X_train_numeric, y_train)

y_pred = gbr.predict(X_val_numeric)

from sklearn.metrics import mean_squared_error
import numpy as np

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print("GBR RMSE:", rmse)

print("GBR R2:", gbr.score(X_val_numeric, y_val))

GBR RMSE: 1048.5853378670286
GBR R2: 0.595458675467171


In [44]:
gbr.score(X_val_numeric, y_val)

0.595458675467171

In [45]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb.fit(X_train_numeric, y_train)

y_pred = xgb.predict(X_val_numeric)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print("XGB RMSE:", rmse)

XGB RMSE: 1060.8603594660128


As Random forrest have the best score, so we are going to predict our test dataset with random forest Ml algorithm

In [46]:
X_test = test_data.drop(['Item_Outlet_Sales'], axis=1)

# Drop same columns as you dropped during training
X_test_numeric = X_test.drop(['Item_Identifier', 'Outlet_Identifier', 'source'], axis=1)

In [55]:
test_pred = rf.predict(X_test_numeric)
print(test_pred)

[1650.59918151 1318.2617486   597.88484562 ... 1891.87523039 3454.98495441
 1347.91431378]


In [56]:
submission = pd.DataFrame({
    'Item_Identifier': test_data['Item_Identifier'],
    'Outlet_Identifier': test_data['Outlet_Identifier'],
    'Item_Outlet_Sales': test_pred
})

In [57]:
submission.to_csv("submission.csv", index=False)

In [58]:
from google.colab import files
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>